# Lab 06 — Não-linearidades estáticas: saturação, zona morta e histerese

**Unidade III — Análise de não-linearidades em malhas de controle** · conteúdo 3.1 do PPC

**Objetivos:**
1. Construir blocos não lineares com `ct.nlsys` e montar malhas com `ct.interconnect`;
2. Quantificar o efeito da **saturação** e da **zona morta** na malha fechada;
3. Simular **histerese** (não-linearidade com memória);
4. Comparar sempre com a malha linear ideal.

**Referências:** Åström & Murray (FBS), caps. 4–5 · Åström & Hägglund, *Advanced PID Control*, cap. 3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

## 1. As três características estáticas

In [ ]:
def sat(u, umax=1.0):
    """Saturação simétrica em +/- umax."""
    return np.clip(u, -umax, umax)

def dead_zone(u, delta=0.5):
    """Zona morta de largura +/- delta."""
    return np.where(np.abs(u) <= delta, 0.0, u - np.sign(u) * delta)

u_in = np.linspace(-3, 3, 500)
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
axs[0].plot(u_in, sat(u_in), lw=2); axs[0].set_title('Saturação (umax = 1)')
axs[1].plot(u_in, dead_zone(u_in), lw=2); axs[1].set_title('Zona morta (δ = 0,5)')

# histerese de relé: depende do sentido de percurso -> laço
u_up = np.linspace(-3, 3, 300)
u_dn = np.linspace(3, -3, 300)
h = 0.5   # meia largura da banda
y_up = np.where(u_up > h, 1.0, -1.0)     # subindo: comuta em +h
y_dn = np.where(u_dn > -h, 1.0, -1.0)    # descendo: comuta em -h
axs[2].plot(u_up, y_up, 'C0', lw=2, label='subindo')
axs[2].plot(u_dn, y_dn, 'C1', lw=2, label='descendo')
axs[2].set_title('Relé com histerese (h = 0,5)'); axs[2].legend()
for ax in axs:
    ax.set_xlabel('entrada'); ax.grid(True)
axs[0].set_ylabel('saída')
plt.show()

## 2. Ferramentas da `python-control` para malhas não lineares

- `ct.nlsys(updfcn, outfcn, ...)`: define um sistema por funções de estado/saída;
- `ct.interconnect([...])`: conecta blocos **pelos nomes dos sinais**;
- `ct.input_output_response(sys, T, U)`: simula a malha no tempo.

Montaremos a malha: referência → **PI** → *não-linearidade do atuador* → **planta** → saída,
com realimentação unitária. Planta: motor CC reduzido $G(s) = \dfrac{5}{2s+1}$.

In [ ]:
# ---- planta linear como sistema E/S ----
G = ct.tf([5], [2, 1])
plant = ct.tf2io(G, inputs='u', outputs='y', name='plant')

# ---- controlador PI (sintonia moderada) como sistema E/S ----
Kp, Ti = 0.8, 1.5
C_pi = ct.tf([Kp * Ti, Kp], [Ti, 0])   # C(s) = Kp (1 + 1/(Ti s))
controller = ct.tf2io(C_pi, inputs='e', outputs='v', name='ctrl')

# ---- somador do erro: e = r - y ----
summer = ct.summing_junction(inputs=['r', '-y'], output='e', name='sum')

# ---- bloco de saturação (sistema estático: sem estados) ----
def sat_out(t, x, u, params):
    return np.clip(u[0], -params['umax'], params['umax'])

def make_sat(umax):
    return ct.nlsys(None, sat_out, inputs='v', outputs='u',
                    params={'umax': umax}, name='sat')

# ---- bloco de zona morta ----
def dz_out(t, x, u, params):
    d = params['delta']
    v = u[0]
    return 0.0 if abs(v) <= d else v - np.sign(v) * d

def make_dz(delta):
    return ct.nlsys(None, dz_out, inputs='v', outputs='u',
                    params={'delta': delta}, name='dz')

# ---- bloco "transparente" para a malha linear ideal ----
ident = ct.nlsys(None, lambda t, x, u, params: u[0],
                 inputs='v', outputs='u', name='ident')

def build_loop(nl_block):
    """Monta a malha r -> [sum] -> [PI] -> [não-linearidade] -> [planta] -> y."""
    return ct.interconnect(
        [summer, controller, nl_block, plant],
        inputs='r', outputs=['y', 'u'])

## 3. Efeito da saturação do atuador

In [ ]:
t = np.linspace(0, 20, 2000)
r = 4.0 * np.ones_like(t)   # degrau de referência "grande" (exige u além do limite)

loop_lin = build_loop(ident)
resp_lin = ct.input_output_response(loop_lin, t, r)

fig, axs = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axs[0].plot(resp_lin.time, resp_lin.outputs[0], 'k--', lw=2, label='linear ideal')
axs[1].plot(resp_lin.time, resp_lin.outputs[1], 'k--', lw=2)

for umax in [2.0, 1.0, 0.6]:
    loop_sat = build_loop(make_sat(umax))
    resp = ct.input_output_response(loop_sat, t, r)
    axs[0].plot(resp.time, resp.outputs[0], lw=2, label=f'umax = {umax}')
    axs[1].plot(resp.time, resp.outputs[1], lw=2)
    axs[1].axhline(umax, color='gray', ls=':', lw=0.8)

axs[0].axhline(4, color='gray', ls='--')
axs[0].set_ylabel('Saída y(t)'); axs[0].legend(); axs[0].grid(True)
axs[1].set_ylabel('Controle u(t)'); axs[1].set_xlabel('Tempo [s]'); axs[1].grid(True)
fig.suptitle('Saturação: resposta mais lenta e sobressinal por windup')
plt.show()

**Diagnóstico:** com saturação severa (umax = 0,6), o controle fica "colado" no limite enquanto
o integrador acumula erro (windup). Quando a saída finalmente cruza a referência, o integrador
carregado empurra além — sobressinal grande e acomodação lenta. O Lab 07 quantifica e o Lab 10
corrige (anti-windup).

## 4. Efeito da zona morta

In [ ]:
t2 = np.linspace(0, 40, 4000)
r2 = np.ones_like(t2)   # degrau unitário

fig, axs = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
resp_lin2 = ct.input_output_response(loop_lin, t2, r2)
axs[0].plot(resp_lin2.time, resp_lin2.outputs[0], 'k--', lw=2, label='linear ideal')
axs[1].plot(resp_lin2.time, resp_lin2.outputs[1], 'k--', lw=2)

for delta in [0.1, 0.3]:
    loop_dz = build_loop(make_dz(delta))
    resp = ct.input_output_response(loop_dz, t2, r2)
    axs[0].plot(resp.time, resp.outputs[0], lw=2, label=f'zona morta δ = {delta}')
    axs[1].plot(resp.time, resp.outputs[1], lw=2)

axs[0].axhline(1, color='gray', ls='--')
axs[0].set_ylabel('Saída y(t)'); axs[0].legend(); axs[0].grid(True)
axs[1].set_ylabel('Controle u(t)'); axs[1].set_xlabel('Tempo [s]'); axs[1].grid(True)
fig.suptitle('Zona morta: partida atrasada e convergência degradada')
plt.show()

A zona morta atrasa a partida (o controle precisa "vencer" δ) e degrada a convergência fina:
perto do regime, o PI produz sinais pequenos que a zona morta anula, e a ação integral fica
rastejando. Em plantas com atrito seco, isso vira oscilação lenta em torno da referência
(*hunting*).

## 5. Histerese: não-linearidade com memória

A histerese exige **estado interno** (a saída anterior). Modelamos um relé com histerese como
`ct.nlsys` **com estado** e o usamos como controlador liga/desliga de um processo térmico
simplificado — o clássico termostato.

In [ ]:
# processo térmico de 1ª ordem: G_term = 2/(50 s + 1)
G_term = ct.tf([2.0], [50.0, 1.0])
plant_t = ct.tf2io(G_term, inputs='u', outputs='y', name='plant')

def relay_update(t, x, u, params):
    """Estado x[0] guarda a saída atual do relé (+1 potência total / 0 desligado)."""
    e = u[0]                       # erro de temperatura
    h = params['h']
    x_new = x[0]
    if e > h:                      # muito frio -> liga
        x_new = 1.0
    elif e < -h:                   # muito quente -> desliga
        x_new = 0.0
    return [x_new]

def relay_output(t, x, u, params):
    return x[0]

relay = ct.nlsys(relay_update, relay_output, states=1, dt=0.1,
                 inputs='e', outputs='u', params={'h': 0.2}, name='relay')

# a planta térmica precisa ser discretizada na mesma taxa do relé
plant_d = ct.tf2io(ct.sample_system(G_term, 0.1), inputs='u', outputs='y', name='plant')
summer_t = ct.summing_junction(inputs=['r', '-y'], output='e', name='sum')

thermostat = ct.interconnect([summer_t, relay, plant_d],
                             inputs='r', outputs=['y', 'u'])

t3 = np.arange(0, 600, 0.1)
r3 = 1.5 * np.ones_like(t3)   # temperatura desejada (normalizada)
resp3 = ct.input_output_response(thermostat, t3, r3, X0=[0, 0])

fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axs[0].plot(resp3.time, resp3.outputs[0], lw=1.5)
axs[0].axhline(1.5, color='gray', ls='--', label='referência')
axs[0].axhline(1.7, color='r', ls=':', lw=0.8, label='banda ±h')
axs[0].axhline(1.3, color='r', ls=':', lw=0.8)
axs[0].set_ylabel('Temperatura'); axs[0].legend(); axs[0].grid(True)
axs[1].step(resp3.time, resp3.outputs[1], lw=1)
axs[1].set_ylabel('Aquecedor (0/1)'); axs[1].set_xlabel('Tempo [s]'); axs[1].grid(True)
fig.suptitle('Controle liga/desliga com histerese: ciclo-limite intencional')
plt.show()

O sistema entra em **ciclo-limite**: oscilação sustentada de amplitude e período fixos.
Aqui ela é intencional (termostato), mas a mesma física aparece indesejada quando há folga em
engrenagens ou banda morta em válvulas. **Meça no gráfico a amplitude e o período** — no Lab 07
esse ciclo-limite será usado para identificar $K_u$ e $T_u$ (experimento do relé).

## 6. Retratos de fase: enxergando a dinâmica não linear inteira de uma vez

Para sistemas de 2ª ordem, o **retrato de fase** ($x_2 \times x_1$ com o campo de vetores)
mostra todos os comportamentos possíveis em um único gráfico — equilíbrios, regiões de
atração e ciclos-limite. É a ferramenta usada no CDS 110 (Caltech) e em FBS cap. 5 para
estudar não-linearidades. Exemplo: pêndulo com realimentação proporcional **saturada** —
a saturação limita o "alcance" do controlador e muda o mapa de regiões de atração:

In [ ]:
pend_params = {'m': 1.0, 'l': 1.0, 'b': 0.5, 'g': 1.0, 'kp': 10.0, 'umax': 1.0}

def pend_cl_update(t, x, u, params):
    """Pendulo invertido com controle P saturado (malha fechada autonoma)."""
    m, l, b, g = map(params.get, ['m', 'l', 'b', 'g'])
    kp, umax = params['kp'], params['umax']
    tau = np.clip(-kp * x[0], -umax, umax)   # controle proporcional saturado
    return np.array([x[1], -b/m * x[1] + (g*l/m) * np.sin(x[0]) + tau/m])

pend_cl = ct.nlsys(pend_cl_update, states=['theta', 'thdot'], inputs=0,
                   outputs=['theta', 'thdot'],
                   params=pend_params, name='pend_cl')

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, umax in zip(axs, [1.0, 3.0]):
    ct.phase_plane_plot(pend_cl, [-2*np.pi, 2*np.pi, -2.5, 2.5], 6,
                        gridspec=[11, 7], params={**pend_params, 'umax': umax},
                        ax=ax, plot_separatrices=False)
    ax.set_title(f'umax = {umax}')
fig.suptitle('Retrato de fase do pêndulo + P saturado: a saturação encolhe a região de atração')
plt.tight_layout(); plt.show()

Com `umax` pequeno, o equilíbrio vertical continua localmente estável, mas trajetórias que
partem longe **escapam** — o atuador não tem força para trazê-las de volta. Aumentar `umax`
amplia a região de atração. **Essa distinção (estabilidade local × global) não existe em
sistemas lineares** e é o motivo de sempre simularmos o sistema não linear completo antes de
ir para a bancada.

---
> **🖼️ Figuras de apoio nos livros:**
> - Ogata, **Figura 8.5** — oscilação sustentada com período $P_{cr}$. Cap. 8, §8.2, **p. 524** (p. 535 do PDF).

## Exercícios (relatório do Lab 06)

**E1.** Meça, para cada `umax` da Seção 3, o sobressinal e o tempo de acomodação. Monte uma
tabela "severidade da saturação × degradação" e comente.

**E2.** Na malha com zona morta δ = 0,3, meça o erro residual médio nos últimos 10 s. A ação
integral zerou o erro? Explique.

**E3.** Implemente a **compensação de zona morta**: antes do bloco `dz`, some
$\hat\delta\,\mathrm{sign}(v)$ ao sinal de controle (use um `ct.nlsys` estático). Teste com
$\hat\delta$ exato e com $\hat\delta$ 50 % maior que o real. O que a supercompensação causa?

**E4.** No termostato, reduza a banda para h = 0,05 e observe a frequência de comutação do
aquecedor. Por que, na prática, não se usa h → 0?

**E5.** No retrato de fase da Seção 6, estime visualmente (e depois confirme simulando com
`ct.input_output_response` a partir de condições iniciais escolhidas) o maior ângulo inicial
$\theta_0$ (com $\dot\theta_0 = 0$) que ainda converge ao equilíbrio para umax = 1.

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui

In [ ]:
# E5 — sua solução aqui